# Setup

In [1]:
import os
import torch
from transformers.utils import logging

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Suppress warnings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

c:\Users\yana\Desktop\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. DistilBERT

## Load

In [2]:
from transformers import AutoTokenizer, AutoModel
from torchinfo import summary

# Load.
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

print(model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

## Summary

In [3]:
from torchinfo import summary

x = tokenizer(
    "I love natural language processing.",
    return_tensors="pt"
)

x = {k: v.to(device) for k, v in x.items()}

model_summary = summary(
    model,
    input_data=x,
    depth=4,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    verbose=0
)

with open("distilbert_summary.txt", "w", encoding="utf-8") as f:
    f.write(str(model_summary))

## Tokenizer

- Normalizer
  - Lowercasing
  - Accent strip
  - Clean text (e.g. \t, \n)
- Pre-tokenizer
  - Split by whitespace or punctuation.
  - Subword tokenization happens within each split.
  - Punctuation is considered as separate split in DistilBERT.
- Subword model
  - WordPiece.
- Post-processor
  - Add special tokens, e.g. [CLS] or [SEP].

### Steps

In [4]:
backend = tokenizer.backend_tokenizer

# Text.
text = "Héllo,\ttransformer!"
print(f"[Text] {text}")

# 1. Normalizer.
normalizer = backend.normalizer
text_norm = normalizer.normalize_str(text)
print(f"[Normalizer] {text_norm}")

# 2. Pre-tokenizer.
pre_tokenizer = backend.pre_tokenizer
text_pre = pre_tokenizer.pre_tokenize_str(text_norm)
print(f"[Pre-tokenizer] {text_pre}")

# 3. Subword model.
subword_model = backend.model
for split, index in text_pre:
    text_sub = subword_model.tokenize(split)
    print(f"[Subword] {split} -> {[token.value for token in text_sub]}")

# 4. Post-processor.
encode = backend.encode(text)
print(f"[Encode] {encode.tokens}")

[Text] Héllo,	transformer!
[Normalizer] hello, transformer!
[Pre-tokenizer] [('hello', (0, 5)), (',', (5, 6)), ('transformer', (7, 18)), ('!', (18, 19))]
[Subword] hello -> ['hello']
[Subword] , -> [',']
[Subword] transformer -> ['transform', '##er']
[Subword] ! -> ['!']
[Encode] ['[CLS]', 'hello', ',', 'transform', '##er', '!', '[SEP]']


### API

In [5]:
from transformers import AutoTokenizer

# Load.
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer_bck = tokenizer.backend_tokenizer

# Structure.
print("Structure:")
print(f"  - [Normalizer] {tokenizer_bck.normalizer}")
print(f"  - [Pre-tokenizer] {tokenizer_bck.pre_tokenizer}")
print(f"  - [Model] {tokenizer_bck.model}")
print(f"  - [Post-processor] {tokenizer_bck.post_processor}", '\n')

# Text.
text = [
    "Hello, transformer!",
]
print(f"[Text] {text}")

# Encode: text -> id.
encode = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=10,
    return_tensors="pt",
)
print(f"[Encode] {encode['input_ids']}")

# Decode: id -> text.
print(f"[Decode] {tokenizer.decode(encode['input_ids'])}")

# token id -> token.
tokens = tokenizer.convert_ids_to_tokens(encode['input_ids'][0])
print(f"[id -> token] {tokens}")

# token -> id.
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"[token -> id] {ids}")

# Methods and attributes.
tokenizer.all_special_tokens    # ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
tokenizer.all_special_ids       # [100, 102, 0, 101, 103]
tokenizer.pad_token
tokenizer.unk_token
tokenizer.bos_token
tokenizer.eos_token

tokenizer.padding_side          # left / right.
tokenizer.truncation_side       # left / right.

tokenizer.get_vocab()           # get vocab.
tokenizer.vocab_size            # vocab size.

tokenizer.model_max_length      # maximum n_tokens of the transformer.

Structure:
  - [Normalizer] BertNormalizer(clean_text=True, handle_chinese_chars=True, strip_accents=None, lowercase=True)
  - [Pre-tokenizer] BertPreTokenizer()
  - [Model] WordPiece(unk_token="[UNK]", continuing_subword_prefix="##", max_input_chars_per_word=100, vocab={"[PAD]":0, "[unused0]":1, "[unused1]":2, "[unused2]":3, "[unused3]":4, ...})
  - [Post-processor] TemplateProcessing(single=[SpecialToken(id="[CLS]", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="[SEP]", type_id=0)], pair=[SpecialToken(id="[CLS]", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="[SEP]", type_id=0), Sequence(id=B, type_id=1), SpecialToken(id="[SEP]", type_id=1)], special_tokens={"[CLS]":SpecialToken(id="[CLS]", ids=[101], tokens=["[CLS]"]), "[SEP]":SpecialToken(id="[SEP]", ids=[102], tokens=["[SEP]"])}) 

[Text] ['Hello, transformer!']
[Encode] tensor([[  101,  7592,  1010, 10938,  2121,   999,   102,     0,     0,     0]])
[Decode] ['[CLS] hello, transformer! [SEP] [PAD] [PAD] [PAD]']


512

## Embedding

- Word embedding: V -> d lookup table, where V = vocab size and d = embedding dimension.
- Positional embedding: another T -> d lookup table, where T = maximum number of tokens of the model.
- Normalization.
  - Each token $x = [x_1, ..., x_{768}]$ normalizes its 768 features.
  - Typical standardization: $x_i' = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$
    - $\mu$ and $\sigma$ are calculated for each token individually, not accross batch or sequence.
  - Learned scale and shift: $y_i = \gamma_i x_i' + \beta_i$

In [6]:
# Encode.
text = [
    "Hello, transformer!",
    "I love you."
]
encode = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=10,
    return_tensors="pt",
)
input_ids = encode['input_ids'].to(device)

# Word embedding.
embed = model.embeddings.word_embeddings
word_emb = embed(input_ids)
print(f"word_emb: {word_emb.shape}")      # (n_seq, n_tokens, d)

# Positional embedding.
position_ids = torch.arange(input_ids.shape[1]).to(device).unsqueeze(0)
embed_pos = model.embeddings.position_embeddings
print(f"pos_emb_shape: {embed_pos.weight.shape}")   # (max_tokens, d)
pos_emb = embed_pos(position_ids)
print(f"pos_emb: {pos_emb.shape}")

# Sum.
emb = word_emb + pos_emb

# Normalization.
embed_norm = model.embeddings.LayerNorm(emb)
print(f"Normalization: {embed_norm.shape}")

# Dropout.
embed_out = model.embeddings.dropout(embed_norm)
print("Dropout: ", embed_out.shape)

word_emb: torch.Size([2, 10, 768])
pos_emb_shape: torch.Size([512, 768])
pos_emb: torch.Size([1, 10, 768])
Normalization: torch.Size([2, 10, 768])
Dropout:  torch.Size([2, 10, 768])


## Transformer

In [7]:
from transformers import AutoModel

model_name = "distilbert/distilbert-base-uncased"
model = AutoModel.from_pretrained(model_name).to(device).eval()
block = model.transformer.layer[0]
block

TransformerBlock(
  (attention): DistilBertSelfAttention(
    (q_lin): Linear(in_features=768, out_features=768, bias=True)
    (k_lin): Linear(in_features=768, out_features=768, bias=True)
    (v_lin): Linear(in_features=768, out_features=768, bias=True)
    (out_lin): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
  (ffn): FFN(
    (dropout): Dropout(p=0.1, inplace=False)
    (lin1): Linear(in_features=768, out_features=3072, bias=True)
    (lin2): Linear(in_features=3072, out_features=768, bias=True)
    (activation): GELUActivation()
  )
  (output_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
)

### Attention

In [8]:
attention = block.attention

q = attention.q_lin(embed_out)      # (2, 10, 768)
k = attention.k_lin(embed_out)
v = attention.v_lin(embed_out)

n_heads = 12
dim_head = int(768 / 12)
q = q.view(2, 10, n_heads, dim_head)    # (2, 10, 12, 64)
k = k.view(2, 10, n_heads, dim_head)
v = v.view(2, 10, n_heads, dim_head)

q = q.transpose(1, 2)       # (2, 12, 10, 64)
k = k.transpose(1, 2)
v = v.transpose(1, 2)

# Attention score.
scores = q @ k.transpose(-2, -1)    # 10x64 @ 64x10 -> 10x10
scores /= 8     # 8 = sqrt(dim_head)

# Padding mask.
attention_mask = encode["attention_mask"].to(device)
mask = attention_mask[:, None, None, :]   # (B, 1, 1, L)

scores = scores.masked_fill(
    mask == 0,
    torch.finfo(scores.dtype).min
)

weights = torch.softmax(scores, dim=-1)
weights = attention.dropout(weights)
print(f"Weights: {weights.shape}")

# Context.
context = weights @ v
context = context.transpose(1, 2)
context = context.reshape(2, 10, 768)
print(f"Context: {context.shape}")

# Out.
attention_out = attention.out_lin(context)

Weights: torch.Size([2, 12, 10, 10])
Context: torch.Size([2, 10, 768])


### Normalization

In [9]:
residual = embed_out + attention_out
out = block.sa_layer_norm(residual)
print(f"Normalization: {out.shape}")

Normalization: torch.Size([2, 10, 768])


### FFN

In [10]:
ffn = block.ffn
x = ffn.lin1(out)
x = ffn.activation(x)   # GELU
x = ffn.lin2(x)
ffn_out = ffn.dropout(x)

### Output LayerNorm

In [11]:
out = block.output_layer_norm(out + ffn_out)

print(f"Final block out: {out.shape}")

Final block out: torch.Size([2, 10, 768])


### Validation

In [12]:
# vs Hugging Face forward.
encoding_gpu = {
    key: value.to(device)
    for key, value in encode.items()
}

with torch.no_grad():
    outputs = model(
        **encoding_gpu,
        output_hidden_states=True
    )

hf_block_out = outputs.hidden_states[1]

print("Manual: ", out.shape)
print("HF:     ", hf_block_out.shape)

print(
    "Same: ",
    torch.allclose(out, hf_block_out, atol=1e-5)
)

print(
    "Max error: ",
    (out - hf_block_out).abs().max().item()
)

Manual:  torch.Size([2, 10, 768])
HF:      torch.Size([2, 10, 768])
Same:  True
Max error:  9.5367431640625e-07


# 2. Qwen2.5-0.5B

## Load

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## Tokenizer

- Normalizer: keeps case and accents.
- Pre-tokenizer: split by whitespace, but whitespaces are together with a word, e.g. `Ġhello`.
- Subword model: byte-level BPE.
- Post-processor: does not append special tokens.

In [14]:
backend = tokenizer.backend_tokenizer

print(f"Normalizer: {backend.normalizer}")
print(f"Pre-tokenizer: {backend.pre_tokenizer}")
print(f"Subword model: {backend.model}", '\n')

text = "HéLLo hello 안녕하세요!"
print(f"Text: {text}")

# Normalizer.
normalizer = backend.normalizer
text_norm = normalizer.normalize_str(text)
print(f"Normalizer: {text_norm}")

# Pre-tokenizer.
pre_tokenizer = backend.pre_tokenizer
text_pre = pre_tokenizer.pre_tokenize_str(text_norm)
print(f"Pre-tokenizer: {text_pre}")

# Subword model, byte-level BPE.
subword_model = backend.model

for split, offset in text_pre:
    tokens = subword_model.tokenize(split)

    print(f"{split} -> {[token.value for token in tokens]}")

# Post-processor.
encode_no_special = tokenizer(text, add_special_tokens=False)
encode_special = tokenizer(text, add_special_tokens=True)

print("No special: ", tokenizer.convert_ids_to_tokens(encode_no_special["input_ids"]))
print("Special: ", tokenizer.convert_ids_to_tokens(encode_special["input_ids"]))

Normalizer: NFC()
Pre-tokenizer: Sequence(pretokenizers=[Split(pattern=Regex("(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+..."), behavior=Isolated, invert=False), ByteLevel(add_prefix_space=False, trim_offsets=True, use_regex=False)])
Subword model: BPE(dropout=None, unk_token=None, continuing_subword_prefix="", end_of_word_suffix="", fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"!":0, """:1, "#":2, "$":3, "%":4, ...}, merges=[("Ġ", "Ġ"), ("ĠĠ", "ĠĠ"), ("i", "n"), ("Ġ", "t"), ("ĠĠĠĠ", "ĠĠĠĠ"), ...]) 

Text: HéLLo hello 안녕하세요!
Normalizer: HéLLo hello 안녕하세요!
Pre-tokenizer: [('HÃ©LLo', (0, 5)), ('Ġhello', (5, 11)), ('ĠìķĪëħķíķĺìĦ¸ìļĶ', (11, 17)), ('!', (17, 18))]
HÃ©LLo -> ['H', 'Ã©', 'LL', 'o']
Ġhello -> ['Ġhello']
ĠìķĪëħķíķĺìĦ¸ìļĶ -> ['ĠìķĪ', 'ëħķ', 'íķĺìĦ¸ìļĶ']
! -> ['!']
No special:  ['H', 'Ã©', 'LL', 'o', 'Ġhello', 'ĠìķĪ', 'ëħķ', 'íķĺìĦ¸ìļĶ', '!']
Special:  ['H', 'Ã©', 'LL', 'o', 'Ġhello', 'ĠìķĪ', 'ëħķ', 'íķĺ

## Embedding

- Vocab size, $V = 151,936$
- Embedding dimension, $d = 896$
- Position embedding is done the later part.

In [15]:
embed = model.model.embed_tokens
print(f"Embedding layer: {embed}", '\n')

# Tokenization.
text = "Hello, transformer!"
encode = tokenizer(
    text,
    return_tensors="pt",
)
input_ids = encode["input_ids"].to(device)

# Embedding.
token_emb = embed(input_ids)

print("Input IDs: ", input_ids.shape)               # (B, L)
print("Embedding table: ", embed.weight.shape)      # (T, d)
print("Token embedding: ", token_emb.shape)         # (B, L, d)

Embedding layer: Embedding(151936, 896) 

Input IDs:  torch.Size([1, 4])
Embedding table:  torch.Size([151936, 896])
Token embedding:  torch.Size([1, 4, 896])


## Transformer

In [16]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

```markdown
embed_tokens, x
    ↓
┌─────────────────────────────────────────────┐
│ Qwen2DecoderLayer 1                         │
│                                             │
│ x                                           │
│  │                                          │
│  ├──────────────────────────────┐           │
│  ↓                              │           │
│ RMSNorm                         │           │
│  ↓                              │           │
│ Q, K, V projections             │           │
│  ↓                              │           │
│ reshape into heads              │           │
│  ↓                              │           │
│ RoPE on Q and K                 │           │
│  ↓                              │           │
│ GQA                             │           │
│  ↓                              │           │
│ QKᵀ / √d                        │           │
│  ↓                              │           │
│ causal mask                     │           │
│  ↓                              │           │
│ softmax                         │           │
│  ↓                              │           │
│ attention weights @ V           │           │
│  ↓                              │           │
│ merge heads                     │           │
│  ↓                              │           │
│ o_proj                          │           │
│  ↓                              │           │
│  +  ←───────────────────────────┘           │
│  │                                          │
│ x'                                          │
│  │                                          │
│  ├────────────────────────────────────┐     │
│  ↓                                    │     │
│ RMSNorm                               │     │
│  ↓                                    │     │
│ gate_proj ─→ SiLU ─┐                  │     │
│                    ├─ element-wise ×  │     │
│ up_proj ───────────┘                  │     │
│  ↓                                    │     │
│ down_proj                             │     │
│  ↓                                    │     │
│  +  ←─────────────────────────────────┘     │
│  │                                          │
│ layer output                                │
└─────────────────────────────────────────────┘
    ↓
Qwen2DecoderLayer 2
    ↓
...
    ↓
Qwen2DecoderLayer 24
    ↓
final RMSNorm
    ↓
lm_head
    ↓
vocabulary logits
```

### RMSNorm

- $RMS(x) = \sqrt{\frac{1}{d} \sum_i x_i^2 + \epsilon}$
- $y_i = \gamma_i \frac{x_i}{RMS(x)}$
- Commonly paired with pre-norm architectures in modern LLMs.
- Slightly less computation, and empirically strong enough.

In [17]:
layer = model.model.layers[0]
rmsnorm = layer.input_layernorm

x_norm = rmsnorm(token_emb)
print(x_norm.shape)

torch.Size([1, 4, 896])


### QKV Projection

- 14 query heads
- 2 key heads
- 2 value heads
- head size = 64

In [18]:
attention = layer.self_attn

print(attention.q_proj)
print(attention.k_proj)
print(attention.v_proj)

q = attention.q_proj(x_norm)
k = attention.k_proj(x_norm)
v = attention.v_proj(x_norm)

# Head sizes.
n_q_heads = 14
n_kv_heads = 2
dim_head = 64

# Reshape.
q = q.view(1, 4, n_q_heads, dim_head)
k = k.view(1, 4, n_kv_heads, dim_head)
v = v.view(1, 4, n_kv_heads, dim_head)

# Transpose: (B, L, H, D) -> (B, H, L, D).
q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)

print("Q: ", q.shape)
print("K: ", k.shape)
print("V: ", v.shape)

Linear(in_features=896, out_features=896, bias=True)
Linear(in_features=896, out_features=128, bias=True)
Linear(in_features=896, out_features=128, bias=True)
Q:  torch.Size([1, 14, 4, 64])
K:  torch.Size([1, 2, 4, 64])
V:  torch.Size([1, 2, 4, 64])


### RoPE

- Rotate Q and K.
$$
\begin{bmatrix}
x'_1 \\
x'_2
\end{bmatrix}
=
\begin{bmatrix}
\cos(p\theta) & -\sin(p\theta) \\
\sin(p\theta) & \cos(p\theta)
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
$$

### GQA, Grouped-Query Attention

- Calculates $QK^T$.
- 1~7 Q share 1st K/V, 8~14 Q share 2nd K/V.

### Causal mask

- Before softmax, mask future tokens.
$$
\begin{bmatrix}
0 & -\infty & -\infty \\
0 & 0 & -\infty \\
0 & 0 & 0
\end{bmatrix}
$$

### SwiGLU MLP

```markdown
x
├─→ Linear 1 → SiLU → gate
└─→ Linear 2        → values

gate * values
      ↓
   Linear 3
      ↓
    output

- gate_proj = Linear 1 = controls how much of each feature passes through.
- up_proj   = Linear 2 = creates features.
- down_proj = Linear 3 = filtered featrues.
```

### LM head

- Each token's embedding -> one score per vocabulary token.
- (B, L, 896) -> LM head -> (B, L, 151,936)

### KV cache

```markdown
Step 1:
"I"
K_I, V_I → cache

Step 2:
"love"
compute Q_love, K_love, V_love
attention uses:
Q_love vs [K_I, K_love]

cache becomes:
[K_I, K_love]
[V_I, V_love]
```